# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Samia2310/flyrank-assignment1-week1/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## My Rule

Pages should be reviewed when they have not been updated for a long time, still receive meaningful search impressions, and have a low click-through rate. These signals suggest that refreshing the content could improve search performance.

### Reason Codes

- stale_but_visible
- stale_content
- low_ctr
- healthy

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [3]:
import pandas as pd

df = pd.read_csv("content_refresh_anonymized.csv")

print(df.shape)
print(df.columns.tolist())
df.head()

(30000, 44)
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 44 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   content_id              30000 non-null  object 
 1   client_id               30000 non-null  object 
 2   search_volume           27532 non-null  float64
 3   competition             27532 non-null  float64
 4   competition_level       27390 non-null  object 
 5   cpc                     27532 non-null  float64
 6   content_type            30000 non-null  object 
 7   main_intent             27626 non-null  object 
 8   word_count              22301 non-null  float64
 9   char_count              22301 non-null  float64
 10  provider_used           8562 non-null   object 
 11  model_used              24267 non-null  object 
 12  impressions_90d         30000 non-null  int64  
 13  clicks_90d              30000 non-null  int64  
 14  pageviews_90d           30000 non-null

In [6]:
import pandas as pd
import os

# Create simple rule features
df["stale"] = (df["days_since_last_update"] >= 180).astype(int)
df["visible"] = (df["impressions_90d"] >= 500).astype(int)
df["low_ctr"] = (df["ctr"] < 1.0).astype(int)

# Baseline score
df["baseline_score"] = (
    df["stale"] * 2 +
    df["visible"] * 2 +
    df["low_ctr"]
)

# Reason code
def reason(row):
    if row["stale"] and row["visible"]:
        return "stale_but_visible"
    elif row["stale"]:
        return "stale_content"
    elif row["low_ctr"]:
        return "low_ctr"
    else:
        return "healthy"

df["reason_code"] = df.apply(reason, axis=1)

# Action label
def action(score):
    if score >= 5:
        return "Refresh Immediately"
    elif score >= 3:
        return "Review Soon"
    else:
        return "Monitor"

df["action"] = df["baseline_score"].apply(action)

# Rank
df = df.sort_values("baseline_score", ascending=False)

# Save CSV
os.makedirs("work/outputs", exist_ok=True)

df.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("CSV saved successfully!")

# Show top 20
top20 = df[[
    "content_id",
    "baseline_score",
    "reason_code",
    "action",
    "impressions_90d",
    "ctr",
    "days_since_last_update"
]].head(20)

top20

CSV saved successfully!


,content_id,baseline_score,reason_code,action,impressions_90d,ctr,days_since_last_update
11489,content_5feee3994adb,5,stale_but_visible,Refresh Immediately,7812,0.01,194
26810,content_ecb6215e79fd,5,stale_but_visible,Refresh Immediately,4429,0.38,194
7021,content_1bfaa38ff26c,5,stale_but_visible,Refresh Immediately,25715,0.23,194
5327,content_fe16a55cd13d,5,stale_but_visible,Refresh Immediately,4556,0.33,194
7452,content_72496874f806,5,stale_but_visible,Refresh Immediately,821,0.24,301
22872,content_e3ff1b093148,5,stale_but_visible,Refresh Immediately,1408,0.28,183
26799,content_77d4d5930e5e,5,stale_but_visible,Refresh Immediately,828,0.24,194
12045,content_c2d929d83eaa,5,stale_but_visible,Refresh Immediately,7558,0.20,193
11630,content_6226ee6adc91,5,stale_but_visible,Refresh Immediately,545,0.18,183
3507,content_074ba6ead17b,5,stale_but_visible,Refresh Immediately,533,0.00,183


In [7]:
top20_review = top20.copy()

top20_review["Confidence"] = "Medium"

top20_review["What would make it wrong"] = (
    "Recent update not reflected, seasonal traffic, or naturally low CTR."
)

top20_review

,content_id,baseline_score,reason_code,action,impressions_90d,ctr,days_since_last_update,Confidence,What would make it wrong
11489,content_5feee3994adb,5,stale_but_visible,Refresh Immediately,7812,0.01,194,Medium,"Recent update not reflected, seasonal traffic,..."
26810,content_ecb6215e79fd,5,stale_but_visible,Refresh Immediately,4429,0.38,194,Medium,"Recent update not reflected, seasonal traffic,..."
7021,content_1bfaa38ff26c,5,stale_but_visible,Refresh Immediately,25715,0.23,194,Medium,"Recent update not reflected, seasonal traffic,..."
5327,content_fe16a55cd13d,5,stale_but_visible,Refresh Immediately,4556,0.33,194,Medium,"Recent update not reflected, seasonal traffic,..."
7452,content_72496874f806,5,stale_but_visible,Refresh Immediately,821,0.24,301,Medium,"Recent update not reflected, seasonal traffic,..."
22872,content_e3ff1b093148,5,stale_but_visible,Refresh Immediately,1408,0.28,183,Medium,"Recent update not reflected, seasonal traffic,..."
26799,content_77d4d5930e5e,5,stale_but_visible,Refresh Immediately,828,0.24,194,Medium,"Recent update not reflected, seasonal traffic,..."
12045,content_c2d929d83eaa,5,stale_but_visible,Refresh Immediately,7558,0.20,193,Medium,"Recent update not reflected, seasonal traffic,..."
11630,content_6226ee6adc91,5,stale_but_visible,Refresh Immediately,545,0.18,183,Medium,"Recent update not reflected, seasonal traffic,..."
3507,content_074ba6ead17b,5,stale_but_visible,Refresh Immediately,533,0.00,183,Medium,"Recent update not reflected, seasonal traffic,..."


## 3. Top-20 review

The ranked queue prioritizes content that is old, still visible in search results, and has relatively low CTR. These pages are the strongest candidates for content refresh.

Confidence is moderate because the rule uses transparent thresholds rather than a trained model.

The ranking could be wrong if:
- the page was recently updated but the dataset has not yet reflected the change,
- seasonal traffic temporarily reduced CTR,
- low CTR is caused by search intent rather than poor content quality,
- the page is intentionally maintained without updates.

In [8]:
# Display the Top-20 review table

top20_review

,content_id,baseline_score,reason_code,action,impressions_90d,ctr,days_since_last_update,Confidence,What would make it wrong
11489,content_5feee3994adb,5,stale_but_visible,Refresh Immediately,7812,0.01,194,Medium,"Recent update not reflected, seasonal traffic,..."
26810,content_ecb6215e79fd,5,stale_but_visible,Refresh Immediately,4429,0.38,194,Medium,"Recent update not reflected, seasonal traffic,..."
7021,content_1bfaa38ff26c,5,stale_but_visible,Refresh Immediately,25715,0.23,194,Medium,"Recent update not reflected, seasonal traffic,..."
5327,content_fe16a55cd13d,5,stale_but_visible,Refresh Immediately,4556,0.33,194,Medium,"Recent update not reflected, seasonal traffic,..."
7452,content_72496874f806,5,stale_but_visible,Refresh Immediately,821,0.24,301,Medium,"Recent update not reflected, seasonal traffic,..."
22872,content_e3ff1b093148,5,stale_but_visible,Refresh Immediately,1408,0.28,183,Medium,"Recent update not reflected, seasonal traffic,..."
26799,content_77d4d5930e5e,5,stale_but_visible,Refresh Immediately,828,0.24,194,Medium,"Recent update not reflected, seasonal traffic,..."
12045,content_c2d929d83eaa,5,stale_but_visible,Refresh Immediately,7558,0.20,193,Medium,"Recent update not reflected, seasonal traffic,..."
11630,content_6226ee6adc91,5,stale_but_visible,Refresh Immediately,545,0.18,183,Medium,"Recent update not reflected, seasonal traffic,..."
3507,content_074ba6ead17b,5,stale_but_visible,Refresh Immediately,533,0.00,183,Medium,"Recent update not reflected, seasonal traffic,..."


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Weak Picks and Leakage Check

Some highly ranked pages may not actually require refreshing because the rule does not consider content quality or business priorities. A page with high impressions but naturally low CTR may still perform as expected.

Leakage Check:
- No future-window information was used.
- No label-derived variables such as trend_direction or trend_pct were included.
- Only historical features available at decision time were used.

In [10]:
# Verify that no leakage columns were used

features_used = [
    "days_since_last_update",
    "impressions_90d",
    "ctr"
]

leakage_features = ["trend_direction", "trend_pct"]

print("Features used:")
print(features_used)

print("\nLeakage features avoided:")
print(leakage_features)

Features used:
['days_since_last_update', 'impressions_90d', 'ctr']

Leakage features avoided:
['trend_direction', 'trend_pct']


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.